# cryosim — coupled burn demo (LOX/LCH4, methane-cooled)

API walk-through of the coupled tank → regen → injector simulation.
See `README.md` for the physics, assumptions, and validation status.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from cryosim.fluids import Fluid
from cryosim.tank_geometry import TankGeometry
from cryosim.thermal_model import TankThermalModel, TankThermalConfig
from cryosim.slosh_model import SloshModel
from cryosim.chamber_geometry import ChamberContour, CoolingChannels
from cryosim.combustion import gas_preset
from cryosim.coupling import BurnProfile, CoupledConfig, CoupledSimulator, Engine

## System definition
0.4 m diameter LCH4 tank (autogenous at 3 bar) feeding a 5 kN-class
LOX/CH4 engine through an electric pump (~90 bar supercritical channels).

In [ ]:
ch4 = Fluid("LCH4")
tank = TankGeometry(radius=0.2, cyl_length=1.0, bottom_dome="elliptical", top_dome="elliptical")
tank_model = TankThermalModel(
    ch4, tank, wall_mass=25.0, heat_flux=150.0,
    config=TankThermalConfig(pressurant_setpoint=3e5, pressurant_max_flow=0.1),
)
engine = Engine(
    contour=ChamberContour(throat_radius=0.019, contraction_ratio=6.0,
                           expansion_ratio=4.5, chamber_length=0.09),
    channels=CoolingChannels(n_channels=60, channel_width=1.0e-3,
                             channel_height=1.4e-3, t_wall=0.7e-3, k_wall=330.0),
    gas=gas_preset("lox/ch4"), mixture_ratio=3.4,
)
sim = CoupledSimulator(
    ch4, tank, tank_model, SloshModel(ch4, tank), engine,
    CoupledConfig(dt=0.25, regen_interval=1.0, pump_dp=87e5),
)

In [ ]:
profile = BurnProfile(
    pc_of_t=lambda t: 30e5,                                   # 30 bar chamber
    axial_accel_of_t=lambda t: (3 + 0.2 * t) * 9.81,          # 3 g -> 7 g
    lateral_accel_of_t=lambda t: 0.8 * np.sin(2 * np.pi * 0.9 * t),
    t_end=20.0,
)
hist = sim.run(profile, P0=3e5, fill0=0.9, T_liquid0=118.0)
a = hist.asarrays()
print(f"fill {a['fill_fraction'][0]:.2f} -> {a['fill_fraction'][-1]:.2f}, "
      f"injector inlet T ends at {a['coolant_outlet_T'][-1]:.0f} K")

## Time histories

In [ ]:
from cryosim import plots
plots.plot_coupled_tank(a, "nb_tank.png")
plots.plot_coupled_slosh(a, "nb_slosh.png")
plots.plot_coupled_regen(a, "nb_regen.png")
t_mid, res_mid = hist.regen_snapshots[len(hist.regen_snapshots) // 2]
plots.plot_regen_distribution(res_mid, engine.contour.x_throat, "nb_regen_axial.png")
for f in ("nb_tank.png", "nb_slosh.png", "nb_regen.png", "nb_regen_axial.png"):
    display(plt.imread(f))  # or open the PNGs directly

## Feed-system recommendation for this configuration

In [ ]:
from cryosim.pid_recommender import FeedSystemConfig, TankSpec, recommend_feed_system, draw_pid
rec = recommend_feed_system(FeedSystemConfig(
    pressurization="pump",
    tanks=[TankSpec("LOX", cryogenic=True, oxidizer=True),
           TankSpec("LCH4", cryogenic=True, is_coolant=True)],
))
print(rec.as_text())
draw_pid(rec, "nb_pid.svg")